In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple, Union
import numpy as np


First Rough Simulation


In [2]:
# =========================================================
# Probability Rule Objects -> Setting up threshold for tau
# =========================================================


@dataclass(frozen=True)
class ThresholdRule:
    """
    Threshold probability rule:
        p(tau) = low  if tau <= threshold
                 high otherwise
    """
    threshold: int
    low: float
    high: float

##Check if value makes sense
    def __post_init__(self) -> None:
        for x in (self.low, self.high):
            if not (0.0 <= x <= 1.0):
                raise ValueError("Probabilities must lie in [0, 1].")
        if self.threshold < 0:
            raise ValueError("Threshold must be nonnegative.")

##Set threshold
    def __call__(self, tau: int) -> float:
        return self.low if tau <= self.threshold else self.high

##Example: rule = ThresholdRule(threshold=2, low=0.1, high=0.4)

In [3]:
# ============================================================
# Model parameter objects -> Setting up patient & model inputs
# ============================================================

ProbabilityFn = Callable[[int], float] #Takes in tau and outputs probability


@dataclass(frozen=True)
class PatientClassParams:
    """
    Parameters for one patient class i.
    """
    class_id: int
    lambda_per_slot: float
    balk_prob: ProbabilityFn          # b_i(tau)
    cancel_prob: float               # phi_i
    no_show_prob: ProbabilityFn      # xi_i(tau)
    value: float = 1.0               # v_i

## Check validity of inputs
    def __post_init__(self) -> None:
        if self.class_id <= 0:
            raise ValueError("class_id must be positive.")
        if self.lambda_per_slot < 0:
            raise ValueError("Arrival rate must be nonnegative.")
        if not (0.0 <= self.cancel_prob <= 1.0):
            raise ValueError("Cancellation probability must lie in [0, 1].")



@dataclass(frozen=True)
class SimulationConfig:
    """
    Global simulation configuration.
    """
    slots_per_day: int
    horizon_days: int
    burn_in_days: int
    measure_days: int
    cooldown_days: int
    classes: Dict[int, PatientClassParams]
    seed: Optional[int] = None

    def __post_init__(self) -> None:
        if self.slots_per_day <= 0:
            raise ValueError("slots_per_day must be positive.")
        if self.horizon_days <= 0:
            raise ValueError("horizon_days must be positive.")
        if self.burn_in_days < 0:
            raise ValueError("burn_in_days must be nonnegative.")
        if self.measure_days <= 0:
            raise ValueError("measure_days must be positive.")
        if self.cooldown_days < 0:
            raise ValueError("cooldown_days must be nonnegative.")
        if not self.classes:
            raise ValueError("At least one patient class is required.")

In [4]:
# ==========================
# State and metrics objects
# ==========================

from dataclasses import dataclass
from typing import Dict, List, Tuple, Union


@dataclass
class Booking:
    """
    One booked appointment.

    booking_delay = tau = original offered booking delay in days
    patient_class = i
    """
    patient_class: int
    booking_delay: int
    tracked: bool


@dataclass
class ClassMetrics:
    """
    Metrics tracked for one patient class.
    """
    arrivals: int = 0
    booked: int = 0
    balked: int = 0
    no_offer: int = 0
    canceled: int = 0
    no_show: int = 0
    served: int = 0
    total_booking_delay: float = 0.0

    @property
    def mean_booking_delay(self) -> float:
        return self.total_booking_delay / self.booked if self.booked > 0 else 0.0

    @property
    def percent_serviced(self) -> float:
        return self.served / self.arrivals if self.arrivals > 0 else 0.0

    def attended_utilization(self, total_slots: int) -> float:
        return self.served / total_slots if total_slots > 0 else 0.0


@dataclass
class SlotMetrics:
    booked_slots: int = 0
    served_slots: int = 0
    no_show_slots: int = 0
    empty_slots: int = 0


@dataclass
class SimulationResults:
    """
    Final simulation outputs.
    """
    class_metrics: Dict[int, ClassMetrics]
    slot_metrics: SlotMetrics
    total_slots: int
    total_value: float
    daily_summary_states: List[Dict[int, List[int]]]
    final_full_state: List[List[Union[int, Tuple[int, int]]]]

    @property
    def total_served(self) -> int:
        return sum(m.served for m in self.class_metrics.values())

    @property
    def overall_percent_serviced(self) -> float:
        total_arrivals = sum(m.arrivals for m in self.class_metrics.values())
        return self.total_served / total_arrivals if total_arrivals > 0 else 0.0

    @property
    def overall_attended_utilization(self) -> float:
        return self.total_served / self.total_slots if self.total_slots > 0 else 0.0

    @property
    def overall_scheduled_utilization(self) -> float:
        return self.slot_metrics.booked_slots / self.total_slots if self.total_slots > 0 else 0.0

In [5]:
# =========================
# Simulation engine
# =========================

class ClinicAppointmentSimulation:
    """
    Slot-by-slot clinic appointment simulation with:
    - 2+ patient classes
    - FCFS booking to earliest open slot strictly after active slot
    - delay-dependent balking
    - delay-dependent no-show
    - constant cancellation, applied only at end of day, and never same-day
    - rolling calendar full state
    - derived summary state at the start of each measured day

    Internal calendar representation:
        self.calendar[r][m] is either None (open slot) or Booking(i, tau, tracked)

    Public full-state view:
        0 or (i, tau)
    """

    def __init__(self, config: SimulationConfig) -> None:
        self.config = config
        self.rng = np.random.default_rng(config.seed)

        # Rolling slot-level calendar Y_t(r, m), stored internally as None or Booking
        self.calendar: List[List[Optional[Booking]]] = [
            [None for _ in range(config.slots_per_day)]
            for _ in range(config.horizon_days)
        ]

        self.class_metrics: Dict[int, ClassMetrics] = {
            class_id: ClassMetrics() for class_id in config.classes
        }
        self.slot_metrics = SlotMetrics()
        self.total_value: float = 0.0

        # One summary state per measured day, recorded at the start of each measured day
        self.daily_summary_states: List[Dict[int, List[int]]] = []

    # -------------------------
    # State views
    # -------------------------

    def full_state_view(self) -> List[List[Union[int, Tuple[int, int]]]]:
        """
        Return Y_t(r, m) with cells shown exactly as:
            0
            (i, tau)
        """
        view: List[List[Union[int, Tuple[int, int]]]] = []
        for day_row in self.calendar:
            row_view: List[Union[int, Tuple[int, int]]] = []
            for cell in day_row:
                if cell is None:
                    row_view.append(0)
                else:
                    row_view.append((cell.patient_class, cell.booking_delay))
            view.append(row_view)
        return view

    def summary_state(self) -> Dict[int, List[int]]:
        """
        Derived summary state at the start of a day:
            X^D_{i,r} = number of class-i patients scheduled for day D+r
        """
        summary: Dict[int, List[int]] = {
            class_id: [0 for _ in range(self.config.horizon_days)]
            for class_id in self.config.classes
        }

        for r in range(self.config.horizon_days):
            for m in range(self.config.slots_per_day):
                cell = self.calendar[r][m]
                if cell is not None:
                    summary[cell.patient_class][r] += 1

        return summary

    # -------------------------
    # Booking logic
    # -------------------------

    def find_earliest_open_slot(self, active_slot: int) -> Optional[Tuple[int, int]]:
        """
        Find the earliest open slot strictly after the current active slot.

        Eligible slots:
        - same day, only m > active_slot
        - future days, any slot
        """
        # Same day: later slots only
        for m in range(active_slot + 1, self.config.slots_per_day):
            if self.calendar[0][m] is None:
                return (0, m)

        # Future days
        for r in range(1, self.config.horizon_days):
            for m in range(self.config.slots_per_day):
                if self.calendar[r][m] is None:
                    return (r, m)

        return None

    def process_one_arrival(
        self,
        class_id: int,
        active_slot: int,
        track_patient: bool,
    ) -> None:
        """
        Process one arriving patient from class i.
        Only measurement-window arrivals are tracked in class metrics.
        """
        params = self.config.classes[class_id]
        metrics = self.class_metrics[class_id]

        if track_patient:
            metrics.arrivals += 1

        offered = self.find_earliest_open_slot(active_slot)

        if offered is None:
            if track_patient:
                metrics.no_offer += 1
            return

        r, m = offered
        tau = r  # offered booking delay in days

        # Balking decision
        if self.rng.random() < params.balk_prob(tau):
            if track_patient:
                metrics.balked += 1
            return

        # Accept and book
        self.calendar[r][m] = Booking(
            patient_class=class_id,
            booking_delay=tau,
            tracked=track_patient,
        )

        if track_patient:
            metrics.booked += 1
            metrics.total_booking_delay += tau

    # -------------------------
    # Service logic
    # -------------------------

    def serve_active_slot(self, active_slot: int, count_slot_metrics: bool) -> None:
        """
        Serve the active slot (r = 0, m = active_slot).

        Slot metrics and value are counted only during measured days.
        Class service/no-show metrics are counted only for tracked patients.
        """
        cell = self.calendar[0][active_slot]

        if cell is None:
            if count_slot_metrics:
                self.slot_metrics.empty_slots += 1
            return

        if count_slot_metrics:
            self.slot_metrics.booked_slots += 1

        class_id = cell.patient_class
        tau = cell.booking_delay
        params = self.config.classes[class_id]
        metrics = self.class_metrics[class_id]

        # No-show decision depends on original tau, not current residual delay
        if self.rng.random() < params.no_show_prob(tau):
            if cell.tracked:
                metrics.no_show += 1
            if count_slot_metrics:
                self.slot_metrics.no_show_slots += 1
        else:
            if cell.tracked:
                metrics.served += 1
            if count_slot_metrics:
                self.slot_metrics.served_slots += 1
                self.total_value += params.value

        # Active slot is consumed after service/no-show
        self.calendar[0][active_slot] = None

    # -------------------------
    # End-of-day logic
    # -------------------------

    def apply_end_of_day_cancellations(self) -> None:
        """
        Apply cancellations only to future appointments with r >= 1.
        Same-day cancellations are not allowed.

        All future bookings may cancel, but only tracked bookings count toward
        class-level cancellation metrics.
        """
        for r in range(1, self.config.horizon_days):
            for m in range(self.config.slots_per_day):
                cell = self.calendar[r][m]
                if cell is None:
                    continue

                class_id = cell.patient_class
                params = self.config.classes[class_id]

                if self.rng.random() < params.cancel_prob:
                    if cell.tracked:
                        self.class_metrics[class_id].canceled += 1
                    self.calendar[r][m] = None

    def roll_calendar_forward_one_day(self) -> None:
        """
        End of day transition:
        - drop day 0
        - shift future days forward by one
        - append a new empty day at the horizon end
        """
        self.calendar.pop(0)
        self.calendar.append([None for _ in range(self.config.slots_per_day)])

    # -------------------------
    # Slot arrivals
    # -------------------------

    def generate_ordered_individual_arrivals(self) -> List[int]:
        """
        Generate class-specific Poisson arrivals for the current slot,
        convert to individual patients, and randomize within-slot order.
        """
        arrivals: List[int] = []

        for class_id, params in self.config.classes.items():
            n = int(self.rng.poisson(params.lambda_per_slot))
            arrivals.extend([class_id] * n)

        if arrivals:
            arrivals = self.rng.permutation(arrivals).tolist()

        return arrivals

    # -------------------------
    # Main run
    # -------------------------

    def run(self) -> SimulationResults:
        """
        Run the slot-by-slot simulation with:
        - burn-in days
        - measurement days
        - cooldown days

        Class metrics track only arrivals from the measurement window.
        Slot metrics and total value count only service occurring on measured days.
        """
        total_days = (
            self.config.burn_in_days
            + self.config.measure_days
            + self.config.cooldown_days
        )

        first_measure_day = self.config.burn_in_days
        last_measure_day_exclusive = self.config.burn_in_days + self.config.measure_days

        for day in range(total_days):
            in_measurement_window = first_measure_day <= day < last_measure_day_exclusive

            # Record summary state only for measured days
            if in_measurement_window:
                start_of_day_summary = self.summary_state()
                self.daily_summary_states.append({
                    class_id: counts.copy()
                    for class_id, counts in start_of_day_summary.items()
                })

            # Process all S active slots in day D
            for s in range(self.config.slots_per_day):
                ordered_arrivals = self.generate_ordered_individual_arrivals()

                for class_id in ordered_arrivals:
                    self.process_one_arrival(
                        class_id=class_id,
                        active_slot=s,
                        track_patient=in_measurement_window,
                    )

                self.serve_active_slot(
                    active_slot=s,
                    count_slot_metrics=in_measurement_window,
                )

            # End-of-day cancellations for r >= 1 only
            self.apply_end_of_day_cancellations()

            # Move to next day
            self.roll_calendar_forward_one_day()

        return SimulationResults(
            class_metrics=self.class_metrics,
            slot_metrics=self.slot_metrics,
            total_slots=self.config.measure_days * self.config.slots_per_day,
            total_value=self.total_value,
            daily_summary_states=self.daily_summary_states,
            final_full_state=self.full_state_view(),
        )

In [ ]:
# =========================
# Example setup and run
# =========================

def build_example_simulation() -> ClinicAppointmentSimulation:
    """
    Example parameterization for 2 patient classes.
    Replace these numbers as needed.
    """
    a = 1.0  # v_1 = v_2 = a

    class_1 = PatientClassParams(
        class_id=1,
        lambda_per_slot=0.80,
        balk_prob=ThresholdRule(threshold=2, low=0.05, high=0.30),
        cancel_prob=0.08,
        no_show_prob=ThresholdRule(threshold=2, low=0.03, high=0.15),
        value=a,
    )

    class_2 = PatientClassParams(
        class_id=2,
        lambda_per_slot=0.55,
        balk_prob=ThresholdRule(threshold=1, low=0.08, high=0.35),
        cancel_prob=0.10,
        no_show_prob=ThresholdRule(threshold=1, low=0.05, high=0.18),
        value=a,
    )

    config = SimulationConfig(
        slots_per_day=8,      # S
        horizon_days=10,      # H
        burn_in_days=30,
        measure_days=60,
        cooldown_days=10,     # often set equal to horizon_days
        classes={
            1: class_1,
            2: class_2,
        },
        seed=42,
    )

    return ClinicAppointmentSimulation(config)


if __name__ == "__main__":
    sim = build_example_simulation()
    results = sim.run()

    print("=== Per-class metrics ===")
    for class_id, m in results.class_metrics.items():
        print(f"\nClass {class_id}")
        print(f"  Arrivals:              {m.arrivals}")
        print(f"  Booked:                {m.booked}")
        print(f"  Balked:                {m.balked}")
        print(f"  NoOffer:               {m.no_offer}")
        print(f"  Canceled:              {m.canceled}")
        print(f"  NoShow:                {m.no_show}")
        print(f"  Served:                {m.served}")
        print(f"  Mean booking delay:    {m.mean_booking_delay:.3f}")
        print(f"  Attended utilization:  {m.attended_utilization(results.total_slots):.3f}")
        print(f"  Percent serviced:      {m.percent_serviced:.3f}")

    print("\n=== Slot metrics ===")
    sm = results.slot_metrics
    print(f"BookedSlots:             {sm.booked_slots}")
    print(f"ServedSlots:             {sm.served_slots}")
    print(f"NoShowSlots:             {sm.no_show_slots}")
    print(f"EmptySlots:              {sm.empty_slots}")

    print("\n=== Aggregate outputs ===")
    print(f"Scheduled utilization:   {results.overall_scheduled_utilization:.3f}")
    print(f"Attended utilization:    {results.overall_attended_utilization:.3f}")
    print(f"Overall percent served:  {results.overall_percent_serviced:.3f}")
    print(f"Total served:            {results.total_served}")
    print(f"Total value:             {results.total_value:.3f}")

    print("\n=== Start-of-day summary state for Day 0 ===")
    # X^D_{i,r}: counts by class i and residual day offset r
    print(results.daily_summary_states[0])

    print("\n=== Final rolling full state Y_t(r,m) ===")
    for r, row in enumerate(results.final_full_state):
        print(f"r={r}: {row}")


=== Per-class metrics ===

Class 1
  Arrivals:              391
  Booked:                351
  Balked:                40
  NoOffer:               0
  Canceled:              46
  NoShow:                11
  Served:                294
  Mean booking delay:    1.695
  Attended utilization:  0.613
  Percent serviced:      0.752

Class 2
  Arrivals:              285
  Booked:                198
  Balked:                87
  NoOffer:               0
  Canceled:              30
  NoShow:                19
  Served:                149
  Mean booking delay:    1.621
  Attended utilization:  0.310
  Percent serviced:      0.523

=== Slot metrics ===
BookedSlots:             474
ServedSlots:             443
NoShowSlots:             31
EmptySlots:              6

=== Aggregate outputs ===
Scheduled utilization:   0.988
Attended utilization:    0.923
Overall percent served:  0.655
Total served:            443
Total value:             443.000

=== Start-of-day summary state for Day 0 ===
{1: [3, 5, 